In [132]:
import json
import os
import time
from pathlib import Path
from typing import Literal
import certifi
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from functools import cache
from langchain_nvidia_ai_endpoints import ChatNVIDIA

In [133]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
GENAI_API_KEY = os.getenv("GOOGLE_API_KEY")
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

if NVIDIA_API_KEY:
    print("NVIDIA API key found")
else:
    print("No NVIDIA API key found. Add NVIDIA_API_KEY to the .env file")
openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

OpenRouter API key found
NVIDIA API key found


In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

response = openrouter.chat.completions.create(model="openrouter/free", messages=messages)
print(f"Answered by {response.model}")
display(Markdown(f"**Response:** {response.choices[0].message.content}"))

In [ ]:
DATA = Path("./data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [ ]:
TINY = True

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

In [ ]:
claims[0]

In [ ]:
labels[claims[0]["claim_id"]]

In [116]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "claim_type": claim["claim_type"],
        "amount_claimed": claim["amount_claimed"],
        "difficulty": label["difficulty"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

,claim_id,summary,claim_type,amount_claimed,difficulty,decision,payout
0,CLM-1007,Phone dropped in the hotel shower,baggage,1099.0,routine,approve,50.0
1,CLM-1015,Flew home from Paris on arrival after leaving ...,trip_interruption,2340.0,routine,decline,0.0
2,CLM-1017,Backpack snatched by a scooter rider in Naples,baggage,830.0,routine,refer,0.0
3,CLM-1020,Prescription sunglasses stolen by a monkey at ...,baggage,240.0,manual,approve,240.0
4,CLM-1021,Tote bag snatched from her shoulder in Buenos ...,baggage,1103.0,hard,approve,825.0


In [117]:
home_alone = [claim for claim in claims if claim["claim_id"] == "CLM-1015"][0]
decision = labels["CLM-1015"]

display(Markdown(f"""**{home_alone['summary']}**

{home_alone['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Flew home from Paris on arrival after leaving a child at home**

We flew from Chicago to Paris on June 20 as a family of six. Somewhere over the Atlantic I realized that our youngest son, Kevin, who is eight, was not on the plane. We had overslept, we left for the airport in a rush and the head count in the van was wrong. I took the first available flight home the moment we landed in Paris. Kevin was fine. In his words, he had handled it. I am claiming my one-way fare home and my share of the prepaid hotel nights in Paris that I did not use.

**Handler:** decline $0.00. Trip Interruption pays only for the reasons listed in the wording. Returning home because a child was left behind is not one of them, and nobody was sick or injured.


In [118]:
# Load the whole queue, in case TINY is True
queue, queue_labels = get_sample_data()
hangover = [claim for claim in queue if claim["claim_id"] == "CLM-1010"][0]
decision = queue_labels["CLM-1010"]

display(Markdown(f"""**{hangover['summary']}**

{hangover['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Emergency dental treatment in Las Vegas for a missing front tooth**

I was in Las Vegas from June 19 for a friend's bachelor party. On the morning of June 21 I woke up on the floor of our suite missing an upper front tooth. I have no memory of the night before, and neither do my friends. I am a dentist, and I have to say that the extraction is consistent with my own technique. I went to an emergency dental clinic that morning for an exam, an X-ray and a temporary bridge, and their bill and notes are attached. There was also a tiger in the bathroom of the suite, which I mention only for completeness. I am not claiming for anything connected with the tiger.

**Handler:** decline $0.00. The treatment was needed, and the bill and the notes are in order. The general exclusions rule out losses while the customer is impaired by alcohol or drugs. The clinic's notes record heavy drinking through the evening, no memory of the night and an extraction that the customer believes he carried out himself during it, so the injury happened while he was impaired and no benefit pays.


In [119]:
fig = px.histogram(data, x="claim_type", color="decision", title="Claims by type")
fig.show()

In [120]:
fig = px.histogram(
    data,
    x="difficulty",
    color="decision",
    category_orders={"difficulty": ["routine", "hard", "manual"]},
    title="Claims by difficulty",
)
fig.show()

In [121]:
class ClaimDecision(BaseModel):
    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

In [122]:

INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS}
{policy_wording}
"""

def make_user_prompt(claim):
    policy = policies[claim["policy_id"]]
    return (
        "# Policy record\n"
        f"{json.dumps(policy, indent=2)}\n\n"
        "# Claim\n"
        f"{json.dumps(claim, indent=2)}\n"
    )

def make_messages(claim):
    """Build the messages for one claim: the task and the wording, then the policy record and the claim."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

In [123]:
@cache
def gemini(model):
    """One structured-output client per model, built once."""
    llm = ChatGoogleGenerativeAI(
        model=model,
        google_api_key=GENAI_API_KEY,
        temperature=0.3,
        timeout=60,
        max_retries=2,
    )
    return llm.with_structured_output(ClaimDecision)

@cache
def nvidia(model):
    client = ChatNVIDIA(
        model=model,
        api_key=NVIDIA_API_KEY,
        temperature=0.5,
        top_p=1,
        timeout=60,
        max_retries=1
    )
    return client.with_structured_output(ClaimDecision)


def failed(claim, model, start, why):
    return {"model": model, "technique": "LLM", "claim_id": claim["claim_id"],
            "decision": None, "payout": 0, "rationale": f"FAILED: {why}"[:300],
            "seconds": round(time.time() - start, 1)}


def decide(claim, model, provider):
    """Ask one model to decide one claim."""
    start = time.time()
    messages = make_messages(claim)
    try:
        if provider == "openrouter":
            response = openrouter.chat.completions.parse(
                model=model,
                messages=messages,
                response_format=ClaimDecision,
                extra_body={"provider": {"require_parameters": True}},
            )
            answer = response.choices[0].message.parsed
        elif provider == "google":
            answer = gemini(model).invoke(messages)
        elif provider == "nvidia":
            answer = decide_nvidia(model, messages)
        else:
            raise ValueError(f"unknown provider: {provider}")
    except ValueError:
        raise
    except Exception as e:  # 429s, 503s, bad JSON: record it and keep the loop going
        return failed(claim, model, start, repr(e))

    if answer is None:
        return failed(claim, model, start, "empty structured output")

    return {
        "model": model,
        "technique": "LLM",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "seconds": round(time.time() - start, 1),
    }

In [124]:
nvidia_client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
    timeout=60,
    max_retries=1,
)

def decide_nvidia(model, messages):
    schema = json.dumps(ClaimDecision.model_json_schema())
    messages = messages + [{
        "role": "user",
        "content": f"Reply with only a JSON object matching this schema, no other text:\n{schema}",
    }]
    response = nvidia_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.3,
        max_tokens=1500,
    )
    text = response.choices[0].message.content.strip()
    text = text.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return ClaimDecision.model_validate_json(text)

In [125]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def run(model, claims, labels, provider):
    """Run the model on the claims and compare against the labels."""
    results = []
    for claim in claims:
        answer = decide(claim, model, provider)
        label = labels[claim["claim_id"]]
        results.append({**answer, "difficulty": label["difficulty"], "right": is_right(answer, label)})
    right = sum(result["right"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{model}: {right} of {len(results)} right in {seconds:.0f} seconds")
    return results

In [136]:
free_results = run("openrouter/free", claims, labels, "openrouter")

openrouter/free: 3 of 5 right in 92 seconds


In [140]:
scored = pd.DataFrame(free_results)
scored.pivot_table(index="model", columns="difficulty", values="right", aggfunc="sum")

difficulty,hard,manual,routine
model,,,
openrouter/free,0,0,3


In [141]:
def show_misses(results, labels):
    """Show each claim in a set of results that the model got wrong, with its rationale and the handler's reason."""
    for result in results:
        if not result["right"]:
            label = labels[result["claim_id"]]
            display(Markdown(f"""**{result['claim_id']}, {label['difficulty']}**: {result['decision']} ${result['payout']:,.2f}, where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {result['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))


show_misses(free_results, labels)

**CLM-1020, manual**: refer $0.00, where the handler said approve $240.00

- Model: The claim is for theft of prescription sunglasses under the Baggage and Personal Effects benefit. The incident occurred during the covered trip, and the amount claimed ($240) is within the per-item limit ($500) and plan maximum ($3,000) for the First plan, with a $0 deductible. However, the policy requires a police report for theft claims (section 5), which is missing. The customer states that the local police did not provide a report due to the nature of the incident (monkey theft). Since a required document is unavailable and the policy does not specify alternatives, the claim must be referred to a colleague for further review.
- Handler: The manual treats a loss to a wild animal as accidental loss. It is not theft, so no police report is needed, and a wild animal is not vermin or a pet. The guide's letter is acceptable evidence. The sunglasses are inside the $500 per-item limit on First and the deductible is $0.
- Tests: A rule found only in the claims manual: losses to wild animals. Under the wording alone it reads as a theft with no police report


**CLM-1021, hard**: approve $1,053.00, where the handler said approve $825.00

- Model: The claim is for theft of baggage and personal effects during the covered trip, which is covered under section 5 of the policy. The theft was reported to the police within 24 hours, and a police report is provided. The items claimed are within the per-item limits: electronics have a per-item limit of $1,200 due to the Electronics Plus add-on, and the tote bag is under the regular per-item limit of $300. The total eligible amount is $1,103. After applying the deductible of $50 for the Premium plan, the payout is $1,053. The claim is within the maximum for Baggage and Personal Effects ($1,500) and was submitted within 90 days.
- Handler: Theft from the person, reported to the police the same afternoon. Electronics Plus raises the per-item limit only for the devices the wording lists: phones, tablets, laptops, cameras and their lenses, e-readers and smartwatches. The e-reader is on the list and is under the limit anyway. Headphones and a game console are not on the list, so each counts at the $300 per-item limit on Premium. The tote is under $100, so the customer's own figure is accepted. The $50 deductible comes off.
- Tests: An add-on with a defined scope: Electronics Plus covers a closed list of devices, and the claim form's category says electronics for items that are not on it


In [142]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

response = llm.invoke(messages)
display(Markdown(response.content))

Arrr, now there be a question to set a sailor's beard aquiver!

The old philosophers, they be sayin' "forty-two" be the answer, but that be a load of bilge — a number's nothin' without a question to sail alongside it, and no one's ever found *that* in a bottle.

Here be my read on it, gleaned from many a long night watch:

- The universe has no purpose stitched into it from the beginnin'. 'Tis a vast dark ocean, and it don't care whether ye make port or run aground.
- That lackin' be no curse, matey. It means ye get to *choose* yer heading. Every soul aboard sets their own compass — be it love, be it craft, be it seein' the horizon one more time.
- Meaning ain't found like buried gold on some marked island. 'Tis *made*, plank by plank, in what ye do and who ye stand beside when the storm hits.

So chart yer own course, mind yer crew, and don't wait for the sea to tell ye why ye sail.

And if ye still want the number, it be 42 — but ye didn't hear that from me. 🏴‍☠️

In [143]:
recorded = {}

@tool
def get_policy(policy_id: str) -> str:
    """Return the policy record for a given policy ID."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"Policy {policy_id} not found"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision for a given claim ID."""
    recorded[claim_id] = {
        "decision": decision,
        "payout": payout,
        "rationale": rationale,
    }
    return f"Recorded decision for {claim_id}: {decision} ${payout:,.2f}. {rationale}"

In [144]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""

def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

            {json.dumps(claim, indent=2)}
            """

agent = create_agent(
    model=llm,
    tools=[get_policy, record_decision],
    system_prompt=AGENT_SYSTEM_PROMPT
)

In [145]:
result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claims[0])}]})

for message in result["messages"][1:]:
    message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_policy (call_29eef7e044f9420fb347d7cc)
 Call ID: call_29eef7e044f9420fb347d7cc
  Args:
    policy_id: POL-2007
================================= Tool Message =================================
Name: get_policy

{
  "policy_id": "POL-2007",
  "customer_id": "CUS-3007",
  "plan": "economy",
  "add_ons": [],
  "purchased_on": "2026-06-02",
  "sold_through": "Northwind Airlines checkout",
  "trip": {
    "destination": "Stockholm, Sweden",
    "departs_on": "2026-07-10",
    "returns_on": "2026-07-17",
    "trip_cost": 2200.0
  },
  "travelers": [
    "Greg Kowalski"
  ],
  "status": "active"
}
================================== Ai Message ==================================
Tool Calls:
  record_decision (call_6f815edc988a4e2195c5c5b6)
 Call ID: call_6f815edc988a4e2195c5c5b6
  Args:
    claim_id: CLM-1007
    decision: approve
    payout: 50
    rationale: Section 5 (Baggage and Personal Effect

In [146]:
def decide_with_agent(claim):
    """Run the agent on one claim and check what it recorded against the handler's decision."""
    start = time.time()
    result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels[claim["claim_id"]]
    return {
        "model": "openrouter/free",
        "technique": "Agent",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "seconds": round(time.time() - start, 1),
    }


recorded.clear()
agent_results = []
for claim in claims:
    agent_results.append(decide_with_agent(claim))

print(f"{len(recorded)} of {len(claims)} claims have a recorded decision")

5 of 5 claims have a recorded decision


In [147]:
pd.DataFrame(free_results + agent_results).groupby(["model", "technique"], sort=False)[["right", "seconds"]].sum()

right  seconds
model           technique                
openrouter/free LLM            3     91.8
                Agent          3    307.1

In [148]:
show_misses(agent_results, labels)

**CLM-1020, manual**: refer $0.00, where the handler said approve $240.00

- Model: The incident occurred during the covered trip and the loss of prescription sunglasses is covered under Baggage and Personal Effects, with a $500 per-item limit and $3,000 maximum, and the deductible is $0, so the $240 claim is within coverage. However, the policy requires a police report of the theft to be filed within 24 hours, which was not obtained as the police refused to file a report. The provided guide's letter does not satisfy this requirement. Therefore the claim is missing a required document and is referred for further review.
- Handler: The manual treats a loss to a wild animal as accidental loss. It is not theft, so no police report is needed, and a wild animal is not vermin or a pet. The guide's letter is acceptable evidence. The sunglasses are inside the $500 per-item limit on First and the deductible is $0.
- Tests: A rule found only in the claims manual: losses to wild animals. Under the wording alone it reads as a theft with no police report


**CLM-1021, hard**: approve $1,053.00, where the handler said approve $825.00

- Model: The claim is for theft of a tote bag containing electronics during the covered trip (July 6, 2026, within policy period July 2-12, 2026). Theft is covered under Baggage and Personal Effects. The police report was filed within 24 hours as required. All claimed items are within per-item limits: electronics ($429, $180, $399) are below the $1,200 limit with Electronics Plus add-on; tote bag ($95) is below the $300 Premium per-item limit. Total eligible amount is $1,103. After applying the $50 deductible for Baggage and Personal Effects, the payout is $1,053, which is below the $1,500 maximum for Premium plan. Required documents (police report, electronics receipts) are provided. No exclusions apply.
- Handler: Theft from the person, reported to the police the same afternoon. Electronics Plus raises the per-item limit only for the devices the wording lists: phones, tablets, laptops, cameras and their lenses, e-readers and smartwatches. The e-reader is on the list and is under the limit anyway. Headphones and a game console are not on the list, so each counts at the $300 per-item limit on Premium. The tote is under $100, so the customer's own figure is accepted. The $50 deductible comes off.
- Tests: An add-on with a defined scope: Electronics Plus covers a closed list of devices, and the claim form's category says electronics for items that are not on it


In [149]:

rows = []
for results in [free_results, agent_results]:
    right = sum(result["right"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    rows.append({
        "LLM": results[0]["model"],
        "technique": results[0]["technique"],
        "accuracy": f"{right} / {len(results)}",
        "speed": f"{seconds / len(results):.1f} seconds a claim",
    })

pd.DataFrame(rows)

,LLM,technique,accuracy,speed
0,openrouter/free,LLM,3 / 5,18.4 seconds a claim
1,openrouter/free,Agent,3 / 5,61.4 seconds a claim
